# 🤖 Zero to Hero: Prompt Engineering for AI Developers
## Part 2 — Advanced Patterns & Production Systems

> **Prerequisite:** Complete Part 1 (Sections 1–6: Setup, Few-shot, CoT, Role Prompting, Structured Output, Prompt Templates)
> **This notebook covers:** Sections 7–10 + Bonus Anti-Patterns + Capstone Project
> **Audience:** Python developers building production AI Agents
> **Time estimate:** 4–6 hours of hands-on practice

---

## 📚 What You'll Master in Part 2

| # | Section | Key Concepts | Difficulty |
|---|---------|-------------|-----------|
| 7 | **ReAct Pattern** | Thought→Action→Observation loop, Tool parsing, Multi-tool agents, Error handling | ⭐⭐⭐ |
| 8 | **Advanced Techniques** | Meta-Prompting, Self-Critique, Reflection, Constitutional AI | ⭐⭐⭐⭐ |
| 9 | **Prompt Evaluation & A/B Testing** | LLM-as-judge, Statistical eval, A/B framework, CI/CD for prompts | ⭐⭐⭐⭐ |
| 10 | **Production Systems** | Versioning, Caching, Cost tracking, Rate limiting, Async | ⭐⭐⭐⭐⭐ |
| 11 | **Anti-Patterns** | 8 critical mistakes and how to fix them | ⭐⭐ |
| 12 | **Capstone Project** | End-to-end AI Code Review Agent combining all techniques | ⭐⭐⭐⭐⭐ |

---

## 🗺️ Full Roadmap Context

```
Phase 1 (Weeks 1–2)
├── LLM APIs (OpenAI · Anthropic · Gemini)
├── Prompt Engineering ← YOU ARE HERE (Part 2)
│   ├── Part 1: Zero/Few-shot · CoT · Role Prompting · Structured Output · Templates
│   └── Part 2: ReAct · Meta-Prompting · Self-Critique · Eval · Production ← THIS NOTEBOOK
├── Tokens & Context
├── Embeddings
└── Structured Output

Phase 2 → Tool Use · ReAct (production) · RAG · Vector DB
Phase 3 → LangChain · LangGraph · CrewAI · Production
Phase 4 → Multi-agent · Planning · Human-in-the-loop
Phase 5 → Safety · Fine-tuning · Frontier Research
```


---
## ⚙️ Setup & Dependencies

Run this cell first. All libraries from Part 1 are required plus a few additions.


In [ ]:
# Install dependencies
!pip install openai tiktoken jinja2 pydantic aiohttp --quiet

import os, json, time, re, asyncio, hashlib, statistics, sqlite3, random
from typing import Optional, Callable, Any
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from datetime import datetime

import tiktoken
from jinja2 import Template
from pydantic import BaseModel, Field
from openai import OpenAI, AsyncOpenAI

# ── API Configuration ──────────────────────────────────────────────────────
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-YOUR_KEY_HERE")
client         = OpenAI(api_key=OPENAI_API_KEY)
async_client   = AsyncOpenAI(api_key=OPENAI_API_KEY)

DEFAULT_MODEL = "gpt-4o-mini"   # Cost-effective for practice
SMART_MODEL   = "gpt-4o"        # Use for complex reasoning tasks

# ── Core helper (same as Part 1) ──────────────────────────────────────────
def chat(
    user_message: str,
    system_message: str = "You are a helpful AI assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.1,
    max_tokens: int = 1000,
    verbose: bool = True
) -> str:
    response = client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )
    result = response.choices[0].message.content
    if verbose:
        u = response.usage
        print(f"Tokens — Input: {u.prompt_tokens} | Output: {u.completion_tokens} | Total: {u.total_tokens}")
        print("─" * 60)
        print(result)
    return result

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

print("✅ Part 2 environment ready!")
print(f"   Default model : {DEFAULT_MODEL}")
print(f"   Smart model   : {SMART_MODEL}")


---
## 🔄 Section 7: ReAct Pattern — The Backbone of AI Agents

### 7.1 Theory: What is ReAct?

**ReAct** (Reasoning + Acting) is a prompting paradigm introduced by Yao et al. (2022) that interleaves *thinking* and *acting* in a synergistic loop. It is the conceptual foundation of virtually every modern AI agent framework — LangChain Agents, LangGraph, AutoGen, CrewAI, and OpenAI's Assistants API all implement some variant of ReAct.

#### The Core Loop

```
User Query
    │
    ▼
┌─────────────────────────────────────────────────────┐
│  Thought:  "I need to find the current GPT-4o price" │
├─────────────────────────────────────────────────────┤
│  Action:   search("GPT-4o pricing 2024")            │
├─────────────────────────────────────────────────────┤
│  Observation: "$2.50/1M input, $10/1M output"       │
└─────────────────────────────────────────────────────┘
         │
         ▼  (repeat until done)
┌─────────────────────────────────────────────────────┐
│  Thought:  "Now I can calculate the monthly cost"   │
├─────────────────────────────────────────────────────┤
│  Action:   calculate("500000 * 30 * 2.50 / 1e6")   │
├─────────────────────────────────────────────────────┤
│  Observation: "37.5"                                │
└─────────────────────────────────────────────────────┘
         │
         ▼
Final Answer: "The monthly cost is $37.50"
```

#### Why ReAct Works Better Than Single-Shot

| Approach | Accuracy (HotpotQA) | Hallucination Rate | Debuggability |
|----------|--------------------|--------------------|---------------|
| Direct answer (no tools) | ~29% | High | Black box |
| Chain-of-Thought only | ~33% | Medium | Hard |
| ReAct (Yao et al. 2022) | ~35–50% | Low | Traceable |
| ReAct + self-reflection | ~55%+ | Very Low | Full trace |

The key insight: **Thought** provides context for the next **Action**, and **Observation** grounds subsequent **Thoughts** in real-world facts, breaking the hallucination cycle.

#### ReAct vs. Pure CoT vs. Tool Use

```
Chain-of-Thought:  Think → Think → Think → Answer      (all internal, can hallucinate facts)
Pure Tool Use:     Act → Act → Act → Answer             (no reasoning between steps)
ReAct:             Think → Act → Observe → Think → Act → ... → Answer  ← Best of both
```

#### Production Considerations

1. **Max steps guard** — Always set a `max_steps` limit to prevent infinite loops
2. **Tool error handling** — Tools can fail; the agent must recover gracefully
3. **Observation truncation** — Long observations must be summarized before re-injection
4. **Parallel tool calls** — Modern APIs (OpenAI function calling) support parallel tool execution
5. **State persistence** — In multi-turn systems, ReAct state must be serialized between sessions

#### When to Use ReAct

| Use ReAct when... | Don't use ReAct when... |
|-------------------|------------------------|
| Task requires external data (search, DB, APIs) | Simple Q&A with known answers |
| Multi-step calculations with real values | Single-tool, single-step tasks |
| Decision depends on runtime state | Latency < 500ms required |
| Audit trail / explainability required | Cost is the primary constraint |


### 7.2 Building the Tool Registry

Before writing the agent loop, we design the **tool layer** — each tool is a Python function that the agent can call by name. In production, these wrap real APIs.


In [ ]:
# ─── 7.2 Tool Registry ──────────────────────────────────────────────────────

import re

# ── Tool 1: Web Search (mock — replace with SerpAPI / Tavily in production) ──
def search_web(query: str) -> str:
    """Simulate web search. In production: use Tavily, SerpAPI, or Bing Search API."""
    MOCK_DB = {
        "gpt-4o pricing":       "GPT-4o: $2.50/1M input tokens, $10.00/1M output tokens (May 2024)",
        "gpt-4o-mini pricing":  "GPT-4o-mini: $0.15/1M input tokens, $0.60/1M output tokens (July 2024)",
        "openai rate limits":   "GPT-4o Tier 5: 10,000 RPM, 30M TPM; Tier 1: 500 RPM, 200K TPM",
        "python asyncio":       "asyncio: Python's async I/O framework; use for concurrent LLM calls",
        "langchain agents":     "LangChain Agents use ReAct under the hood; AgentExecutor manages the loop",
        "vector database":      "Vector DBs (Chroma, Pinecone, Weaviate) store embeddings for semantic search",
        "claude pricing":       "Claude 3.5 Sonnet: $3/1M input, $15/1M output",
    }
    query_lower = query.lower()
    for key, val in MOCK_DB.items():
        if key in query_lower:
            return val
    return f"Search results for '{query}': [Top result discusses {query} in the context of AI development]"


# ── Tool 2: Calculator ────────────────────────────────────────────────────────
def calculate(expression: str) -> str:
    """Safely evaluate a math expression. Blocks all builtins to prevent injection."""
    try:
        safe_globals = {"__builtins__": {}, "abs": abs, "round": round, "min": min, "max": max}
        result = eval(expression, safe_globals)
        return f"{expression} = {result:,}" if isinstance(result, (int, float)) else str(result)
    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


# ── Tool 3: Unit Converter ────────────────────────────────────────────────────
def convert_units(conversion: str) -> str:
    """Convert between common units. E.g., '100 USD to tokens'"""
    conversions = {
        ("tokens", "usd"): lambda x: f"{x:,} tokens ≈ ${x / 1_000_000 * 2.50:.4f} (GPT-4o input)",
        ("usd", "tokens"): lambda x: f"${x} ≈ {int(x / 2.50 * 1_000_000):,} input tokens (GPT-4o)",
        ("gb", "mb"):      lambda x: f"{x} GB = {x * 1024:.0f} MB",
    }
    conversion_lower = conversion.lower()
    for (from_u, to_u), fn in conversions.items():
        if from_u in conversion_lower and to_u in conversion_lower:
            nums = re.findall(r"[\d.]+", conversion)
            if nums:
                return fn(float(nums[0]))
    return f"Conversion result for '{conversion}': [specify 'X unit to unit' format]"


# ── Tool 4: Code Analyzer ─────────────────────────────────────────────────────
def analyze_code(code_snippet: str) -> str:
    """Analyze a Python code snippet for complexity, issues, and patterns."""
    lines = code_snippet.strip().split("
")
    functions = [l.strip() for l in lines if l.strip().startswith("def ")]
    classes   = [l.strip() for l in lines if l.strip().startswith("class ")]
    imports   = [l.strip() for l in lines if l.strip().startswith(("import", "from"))]
    has_loop  = any("for " in l or "while " in l for l in lines)
    has_try   = any("try:" in l for l in lines)
    return (
        f"Code Analysis: {len(lines)} lines | Functions: {len(functions)} | "
        f"Classes: {len(classes)} | Imports: {len(imports)} | "
        f"Has loops: {has_loop} | Has error handling: {has_try} | "
        f"Functions: {[f.split('(')[0].replace('def ','') for f in functions]}"
    )


# ── Tool Registry ─────────────────────────────────────────────────────────────
TOOLS = {
    "search":       search_web,
    "calculate":    calculate,
    "convert":      convert_units,
    "analyze_code": analyze_code,
}

TOOL_DESCRIPTIONS = """
Available tools — call EXACTLY one per Action step:
- search(query: str)              → Search the web for current information
- calculate(expression: str)      → Evaluate math: calculate("500_000 * 30 * 2.50 / 1_000_000")
- convert(conversion: str)        → Unit conversion: convert("1M tokens to USD")
- analyze_code(code_snippet: str) → Analyze Python code snippet
"""

# Quick sanity tests
print("Tool registry loaded. Running sanity checks...")
print(f"  search:    {search_web('gpt-4o pricing')[:60]}...")
print(f"  calculate: {calculate('(500_000 * 1000 * 2.50) / 1_000_000')}")
print(f"  convert:   {convert_units('100 USD to tokens')}")
print("✅ All tools operational!")


### 7.3 Designing the ReAct System Prompt

The system prompt is the **contract** between you and the agent. It must be:
- **Unambiguous** — the format must be parseable by regex without ambiguity
- **Complete** — all tools described with their exact call syntax  
- **Constrained** — explicit rules prevent hallucination and infinite loops


In [ ]:
# ─── 7.3 ReAct System Prompt Engineering ────────────────────────────────────

REACT_SYSTEM_PROMPT = f"""You are an expert AI assistant that solves problems step-by-step using tools.

STRICT FORMAT — follow this EXACTLY on every response:

Thought: [Your internal reasoning. What do you know? What do you need? What is your plan?]
Action: tool_name("parameters")

OR when you have enough information to answer:

Thought: [Final reasoning summarizing all gathered facts]
Final Answer: [Complete, well-formatted answer to the user's question]

{TOOL_DESCRIPTIONS}

RULES:
1. Always start with "Thought:" — never skip this step
2. One Action per response — do not chain multiple actions in one turn
3. Use EXACT tool names from the list above
4. String parameters must be in double quotes: search("query here")
5. After each Observation, write a new Thought before the next Action
6. Never hallucinate tool results — wait for the Observation
7. Reach Final Answer as soon as you have sufficient information
8. If a tool returns an error, try a different approach or tool
"""

print("ReAct System Prompt preview:")
print(REACT_SYSTEM_PROMPT[:500] + "...")
print(f"
Prompt token count: {count_tokens(REACT_SYSTEM_PROMPT)} tokens")


### 7.4 Action Parser — The Bridge Between LLM Output and Tool Calls

The parser is critical infrastructure. A fragile parser = a fragile agent. We handle multiple output formats the LLM might generate.


In [ ]:
# ─── 7.4 Robust Action Parser ────────────────────────────────────────────────

def parse_react_action(text: str) -> tuple[Optional[str], Optional[str]]:
    """
    Parse 'Action: tool_name("params")' from ReAct LLM output.

    Handles multiple formats the LLM might generate:
      - Action: search("GPT-4o pricing")         <- standard double quotes
      - Action: search('GPT-4o pricing')          <- single quotes
      - Action: calculate(500 * 30)               <- no quotes (math expression)
      - Action: search(query="GPT-4o pricing")    <- keyword argument

    Returns: (tool_name, tool_input) or (None, None) if not found.
    """
    patterns = [
        r'Action:\s*(\w+)\(["'`](.*?)["'\`]\)',     # Standard: tool("param")
        r'Action:\s*(\w+)\(([^)]+)\)',                  # No quotes: tool(expr)
        r'Action:\s*(\w+):\s*["'\`]?(.*?)["'\`]?$', # Alternate: tool: param
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            tool_name  = match.group(1).strip().lower()
            tool_input = match.group(2).strip().strip(""'`")
            # Handle keyword args like query="..."
            kw_match = re.match(r'\w+=["'`]?(.*)["'\`]?', tool_input)
            if kw_match:
                tool_input = kw_match.group(1).strip(""'`")
            return tool_name, tool_input

    return None, None


# ── Test the parser with various formats ──────────────────────────────────────
test_cases = [
    ('Action: search("GPT-4o pricing")',                    ("search", "GPT-4o pricing")),
    ("Action: calculate(500_000 * 30 * 2.50 / 1_000_000)", ("calculate", "500_000 * 30 * 2.50 / 1_000_000")),
    ("Action: convert('1M tokens to USD')",                  ("convert", "1M tokens to USD")),
    ('Action: search(query="openai rate limits")',           ("search", "openai rate limits")),
]

print("Parser Test Suite:")
all_pass = True
for text, expected in test_cases:
    result = parse_react_action(text)
    status = "✅ PASS" if result == expected else "❌ FAIL"
    if result != expected:
        all_pass = False
    print(f"  {status}: '{text}'")
    if result != expected:
        print(f"    Got: {result} | Expected: {expected}")

print("
✅ All parser tests passed!" if all_pass else "
⚠️ Some parser tests failed")


### 7.5 The ReAct Agent Loop — Full Implementation

The agent loop orchestrates everything: LLM call → parse output → dispatch to tool → inject observation → repeat → terminate when done.


In [ ]:
# ─── 7.5 Full ReAct Agent Loop ───────────────────────────────────────────────

@dataclass
class AgentStep:
    """Represents one step in the ReAct trace for debugging and auditing."""
    step_number:  int
    thought:      str
    action:       Optional[str]
    action_input: Optional[str]
    observation:  Optional[str]
    is_final:     bool = False
    final_answer: Optional[str] = None
    latency_ms:   float = 0.0


def extract_thought(text: str) -> str:
    """Extract the Thought section from agent output."""
    match = re.search(r'Thought:\s*(.*?)(?=Action:|Final Answer:|$)', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def run_react_agent(
    user_query: str,
    max_steps: int = 8,
    verbose: bool = True,
    model: str = SMART_MODEL
) -> dict:
    """
    Full ReAct agent loop with step-by-step trace, error handling,
    token usage tracking, and latency measurement.

    Returns:
        final_answer:     str
        steps:            list[AgentStep]
        total_tokens:     int
        total_latency_ms: float
        success:          bool
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user",   "content": user_query}
    ]

    steps          = []
    total_tokens   = 0
    total_latency  = 0.0

    if verbose:
        print(f"ReAct Agent | model={model}")
        print(f"Query: {user_query}")
        print("=" * 70)

    for step_num in range(1, max_steps + 1):
        start_time = time.time()

        # ── LLM Call ──────────────────────────────────────────────────────────
        try:
            response = client.chat.completions.create(
                model=model, temperature=0.1, max_tokens=600, messages=messages
            )
        except Exception as e:
            print(f"LLM API error at step {step_num}: {e}")
            break

        latency_ms    = (time.time() - start_time) * 1000
        total_latency += latency_ms
        total_tokens  += response.usage.total_tokens
        agent_output   = response.choices[0].message.content

        if verbose:
            print(f"
--- Step {step_num}  [{latency_ms:.0f}ms | {response.usage.total_tokens} tokens]")
            print(agent_output)

        # ── Check for Final Answer ─────────────────────────────────────────────
        if "Final Answer:" in agent_output:
            final_answer = agent_output.split("Final Answer:")[-1].strip()
            steps.append(AgentStep(
                step_number=step_num, thought=extract_thought(agent_output),
                action=None, action_input=None, observation=None,
                is_final=True, final_answer=final_answer, latency_ms=latency_ms
            ))
            if verbose:
                print(f"
{'=' * 70}")
                print(f"Agent completed in {step_num} steps | "
                      f"Total tokens: {total_tokens:,} | Latency: {total_latency:.0f}ms")
            return {
                "final_answer": final_answer, "steps": steps,
                "total_tokens": total_tokens, "total_latency_ms": total_latency, "success": True
            }

        # ── Parse and Execute Action ───────────────────────────────────────────
        tool_name, tool_input = parse_react_action(agent_output)
        thought = extract_thought(agent_output)

        if tool_name and tool_name in TOOLS:
            try:
                observation = TOOLS[tool_name](tool_input)
                if len(observation) > 500:
                    observation = observation[:497] + "..."  # Prevent context explosion
            except Exception as e:
                observation = f"Tool error: {e}. Try a different approach."
        else:
            observation = (f"Unknown tool '{tool_name}'. "
                           f"Available: {list(TOOLS.keys())}. Use a valid tool name.")

        if verbose:
            print(f"
Observation: {observation}")

        steps.append(AgentStep(
            step_number=step_num, thought=thought,
            action=tool_name, action_input=tool_input,
            observation=observation, latency_ms=latency_ms
        ))

        # ── Inject Observation Back into Conversation ─────────────────────────
        messages.append({"role": "assistant", "content": agent_output})
        messages.append({
            "role": "user",
            "content": f"Observation: {observation}

Continue with the next Thought:"
        })

    if verbose:
        print(f"
Max steps ({max_steps}) reached without Final Answer.")
    return {
        "final_answer": "Could not complete within step limit.",
        "steps": steps, "total_tokens": total_tokens,
        "total_latency_ms": total_latency, "success": False
    }


print("✅ ReAct agent loop defined!")


### 7.6 Practice: Run the ReAct Agent on Real Queries


In [ ]:
# ─── 7.6a Single-Tool Test ───────────────────────────────────────────────────

print("TEST 1: Single-tool query (calculator)")
result = run_react_agent(
    "If I run 100,000 GPT-4o API calls per day for a month (30 days), "
    "with each call using 800 input tokens and 400 output tokens, "
    "what is the total cost in USD?",
    max_steps=5
)
print(f"
Final Answer: {result['final_answer']}")


In [ ]:
# ─── 7.6b Multi-Tool Test ────────────────────────────────────────────────────

print("TEST 2: Multi-tool query (search + calculate)")
result = run_react_agent(
    "Compare the monthly cost of running 1 million API calls/day using GPT-4o-mini "
    "versus GPT-4o, assuming 500 input tokens and 200 output tokens per call. "
    "Which is more cost-effective and by what factor?",
    max_steps=8
)
print(f"
Final Answer: {result['final_answer']}")
print(f"
Agent Stats:")
print(f"  Steps taken  : {len(result['steps'])}")
print(f"  Total tokens : {result['total_tokens']:,}")
print(f"  Latency      : {result['total_latency_ms']:.0f}ms")


In [ ]:
# ─── 7.6c Agent Trace Visualization ─────────────────────────────────────────

def print_agent_trace(result: dict):
    """Pretty-print the full ReAct trace for debugging."""
    print("
" + "=" * 70)
    print("FULL AGENT TRACE")
    print("=" * 70)
    for step in result['steps']:
        print(f"
[Step {step.step_number}] {step.latency_ms:.0f}ms")
        if step.thought:
            print(f"  Thought    : {step.thought[:150]}...")
        if step.action:
            print(f"  Action     : {step.action}('{step.action_input}')")
        if step.observation:
            print(f"  Observation: {step.observation[:120]}...")
        if step.is_final:
            print(f"  Final Answer: {step.final_answer[:200]}")

print_agent_trace(result)


### 7.7 Advanced: Error Handling & Retry Logic

Production agents must be resilient. Tools fail, LLMs hallucinate tool names, network timeouts happen.


In [ ]:
# ─── 7.7 ReAct with Automatic Retry & Model Fallback ────────────────────────

def run_react_agent_with_retry(
    user_query: str,
    max_steps: int = 8,
    max_retries: int = 2,
    fallback_model: str = DEFAULT_MODEL
) -> dict:
    """
    ReAct agent with automatic retry on failure.
    Strategy:
      1. Try with SMART_MODEL (gpt-4o)
      2. On failure, retry with DEFAULT_MODEL (gpt-4o-mini — cheaper, faster)
      3. On second failure, return graceful error response
    """
    models_to_try = [SMART_MODEL] + [fallback_model] * max_retries

    for attempt, model in enumerate(models_to_try, 1):
        print(f"
Attempt {attempt} (model: {model})")
        result = run_react_agent(user_query, max_steps=max_steps, verbose=False, model=model)

        if result["success"]:
            result["attempt"]    = attempt
            result["model_used"] = model
            return result

        print(f"  Attempt {attempt} failed. "
              f"{'Retrying...' if attempt < len(models_to_try) else 'All attempts exhausted.'}")

    return {
        "final_answer": "Service temporarily unavailable. Please try again.",
        "success": False, "attempt": len(models_to_try)
    }


# Test resilience
print("Testing retry mechanism...")
result = run_react_agent_with_retry(
    "What is the square root of 144 plus the square root of 169?",
    max_steps=4
)
print(f"
Success on attempt {result.get('attempt', 'N/A')}")
print(f"Model used: {result.get('model_used', 'N/A')}")
print(f"Answer: {result['final_answer']}")


### 7.8 Practice Exercises — ReAct

**Exercise 7A:** Add a new tool `get_datetime(timezone: str)` that returns the current date/time in a given timezone (use `zoneinfo`). Register it in `TOOLS`, update `REACT_SYSTEM_PROMPT`, and test with the query: *"What time is it in Tokyo and New York right now?"*

**Exercise 7B:** Implement a `database_query(sql: str)` tool that queries an in-memory SQLite database seeded with AI model pricing data. Ask the agent: *"Which model gives the best input tokens per dollar?"*

**Exercise 7C:** Add tool-call loop detection to `run_react_agent`. If the same tool is called with the same input twice, return a "Tool loop detected — try different approach" observation and force the agent to change strategy.


In [ ]:
# ─── Exercise 7A Starter: get_datetime tool ──────────────────────────────────
import zoneinfo

def get_datetime(timezone: str = "UTC") -> str:
    """Return current date and time in the specified timezone."""
    try:
        from datetime import datetime
        tz  = zoneinfo.ZoneInfo(timezone)
        now = datetime.now(tz)
        return f"Current time in {timezone}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except Exception as e:
        return f"Error: {e}. Use IANA timezone names like 'Asia/Ho_Chi_Minh', 'America/New_York'."

# Test the function
print(get_datetime("Asia/Ho_Chi_Minh"))
print(get_datetime("America/New_York"))
print(get_datetime("UTC"))

# TODO: Add to TOOLS dict and update REACT_SYSTEM_PROMPT, then run:
# TOOLS["get_datetime"] = get_datetime
# result = run_react_agent("What time is it in Tokyo and New York right now?")


In [ ]:
# ─── Exercise 7B Starter: SQLite database tool ───────────────────────────────

# Seed an in-memory database with AI model pricing
conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE models (
        name TEXT, provider TEXT,
        input_cost_per_1m REAL, output_cost_per_1m REAL,
        context_window_k INTEGER, tokens_per_second INTEGER
    )
""")
conn.executemany("INSERT INTO models VALUES (?,?,?,?,?,?)", [
    ("gpt-4o",           "OpenAI",    2.50,  10.00, 128, 80),
    ("gpt-4o-mini",      "OpenAI",    0.15,   0.60, 128, 120),
    ("claude-3-5-sonnet","Anthropic", 3.00,  15.00, 200, 75),
    ("claude-3-haiku",   "Anthropic", 0.25,   1.25, 200, 150),
    ("gemini-1.5-pro",   "Google",    1.25,   5.00, 1000, 90),
    ("gemini-1.5-flash", "Google",    0.075,  0.30, 1000, 200),
])
conn.commit()

def database_query(sql: str) -> str:
    """Execute a read-only SQL query. Schema: models(name, provider, input_cost_per_1m, output_cost_per_1m, context_window_k, tokens_per_second)"""
    if not sql.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT queries are allowed."
    try:
        cursor = conn.execute(sql)
        cols   = [d[0] for d in cursor.description]
        rows   = cursor.fetchall()
        if not rows: return "No results found."
        result = " | ".join(cols) + "
"
        result += "
".join(" | ".join(str(v) for v in row) for row in rows[:10])
        return result
    except Exception as e:
        return f"SQL Error: {e}"

print("Database seeded. Testing query:")
print(database_query("SELECT name, input_cost_per_1m FROM models ORDER BY input_cost_per_1m"))

# TODO: Add to TOOLS and test with the agent
# TOOLS["database_query"] = database_query


---
## 🔬 Section 8: Advanced Techniques — Meta-Prompting, Self-Critique & Constitutional AI

### 8.1 Theory: The Three Pillars of Advanced Prompt Engineering

#### Pillar 1 — Meta-Prompting: LLMs Writing Better Prompts Than Humans

Meta-Prompting treats the LLM itself as a prompt engineer. Instead of manually crafting prompts, you describe what you want in plain language and let the model generate an optimized prompt.

**Why this works:** LLMs have seen millions of high-quality prompts in their training data. They can exploit their own strengths and quirks better than most humans.

```
Traditional:   Human writes prompt  →  LLM responds
Meta-prompt:   Human describes task  →  LLM writes optimal prompt  →  LLM responds
```

Research (Zhou et al., 2022): Meta-Prompting outperforms human-written prompts on BIG-Bench tasks by **5–15%** when the meta-prompt is well-designed.

#### Pillar 2 — Self-Critique & Reflection

Self-Critique enables the model to evaluate and improve its own outputs. This mirrors how experts work: produce → review → revise.

```
Step 1 Generate : First draft (may have errors, gaps, hallucinations)
Step 2 Critique : Identify specific issues against explicit criteria
Step 3 Revise   : Fix each issue in a complete rewrite
Step 4 Validate : Confirm improvement with metrics (optional)
```

Key insight: Using a **stronger model** or **different temperature** for critique than for generation produces better results, because it activates a different mode of processing.

#### Pillar 3 — Constitutional AI

Constitutional AI (Anthropic, 2022) extends self-critique by evaluating against a set of explicit **principles** (a "constitution"). Rather than open-ended critique, the model checks: "Does this response follow principle #1? #2? #3?"

| Technique | Best for | Cost multiplier | Quality gain |
|-----------|----------|----------------|-------------|
| Single-shot | Speed-sensitive tasks | 1x | baseline |
| Self-Critique (2-step) | Quality-sensitive tasks | ~2.5x | +15–25% |
| Self-Critique (3-step) | High-stakes content | ~4x | +25–40% |
| Constitutional (5+ principles) | Safety/compliance | ~5x | +30–50% |
| Meta-Prompting | Prompt optimization | ~2x | +5–15% |


### 8.2 Meta-Prompting — Automated Prompt Generation


In [ ]:
# ─── 8.2 Meta-Prompting Framework ───────────────────────────────────────────

META_PROMPT_TEMPLATE = """You are a world-class Prompt Engineer with expertise in {model} capabilities.

Generate the OPTIMAL system prompt for this use case:

## Use Case Specification
- Task:              {task}
- Target Model:      {model}
- Output Format:     {output_format}
- User Persona:      {user_persona}
- Constraints:       {constraints}
- Quality Criteria:  {quality_criteria}

## Required Prompt Components — include ALL of these:
1. Specific role definition (not generic)
2. Step-by-step task instructions in positive language
3. Exact output format specification
4. 3 diverse inline examples covering edge cases
5. Error handling for ambiguous or out-of-scope inputs
6. Quality guardrails (length, tone, precision)

## Output Format
Write the complete system prompt between <prompt> tags.
Then explain design decisions in <reasoning> with 5 bullet points.
"""


def meta_prompt(
    task: str,
    output_format: str,
    user_persona: str = "software developers",
    constraints: str = "Response under 500 tokens",
    quality_criteria: str = "Accurate, specific, actionable",
    model: str = "GPT-4o"
) -> dict:
    """
    Use an LLM to generate an optimized prompt for a given task.
    Returns: {'generated_prompt': str, 'reasoning': str, 'token_count': int}
    """
    meta_input = META_PROMPT_TEMPLATE.format(
        task=task, output_format=output_format, user_persona=user_persona,
        constraints=constraints, quality_criteria=quality_criteria, model=model
    )

    response = client.chat.completions.create(
        model=SMART_MODEL, temperature=0.4, max_tokens=1500,
        messages=[{"role": "user", "content": meta_input}]
    )
    output = response.choices[0].message.content

    # Extract sections
    prompt_match    = re.search(r'<prompt>(.*?)</prompt>', output, re.DOTALL)
    reasoning_match = re.search(r'<reasoning>(.*?)</reasoning>', output, re.DOTALL)

    generated_prompt = prompt_match.group(1).strip()    if prompt_match    else output
    reasoning        = reasoning_match.group(1).strip() if reasoning_match else ""

    print("META-PROMPT RESULT")
    print("=" * 70)
    print("
GENERATED SYSTEM PROMPT:")
    print("-" * 70)
    print(generated_prompt)
    print("
DESIGN REASONING:")
    print("-" * 70)
    print(reasoning)
    print(f"
Prompt token count: {count_tokens(generated_prompt)}")

    return {
        "generated_prompt": generated_prompt,
        "reasoning": reasoning,
        "token_count": count_tokens(generated_prompt)
    }


# Demo: Generate a prompt for security code review
meta_result = meta_prompt(
    task="Review Python code for security vulnerabilities and suggest fixes",
    output_format="JSON: {vulnerabilities: [{line, severity: low|medium|high|critical, description, fix}], overall_score: 0-10, summary: str}",
    user_persona="Senior Python developers and security engineers",
    constraints="Identify at least SQL injection, XSS, hardcoded secrets; response under 800 tokens",
    quality_criteria="Precise line-level feedback; zero false negatives for critical vulnerabilities"
)


In [ ]:
# ─── 8.2b Use the Meta-Generated Prompt ─────────────────────────────────────

# Test the auto-generated prompt on real vulnerable code
vulnerable_code = '''
import sqlite3, hashlib

SECRET_KEY = "my_super_secret_key_123"  # Hardcoded!

def get_user(username, password):
    conn = sqlite3.connect("users.db")
    query = f"SELECT * FROM users WHERE username = '{username}'"  # SQL injection
    return conn.execute(query).fetchone()

def render_profile(user_input):
    return f"<div>Welcome, {user_input}!</div>"   # XSS

def authenticate(pwd):
    return hashlib.md5(pwd.encode()).hexdigest()   # Weak hash
'''

print("Testing meta-generated prompt on vulnerable code:")
print("-" * 70)
security_review = chat(
    vulnerable_code,
    system_message=meta_result["generated_prompt"],
    model=SMART_MODEL,
    max_tokens=1000
)


### 8.3 Self-Critique Pipeline — 3-Step Reflection


In [ ]:
# ─── 8.3 Self-Critique & Reflection Pipeline ─────────────────────────────────

@dataclass
class CritiqueResult:
    task:           str
    initial_output: str
    critique:       str
    improved_output:str
    overall_score:  float


def self_critique_pipeline(
    task: str,
    criteria: list[str],
    generation_model: str = DEFAULT_MODEL,
    critique_model:   str = SMART_MODEL,
    revision_model:   str = SMART_MODEL,
    verbose: bool = True
) -> CritiqueResult:
    """
    3-step self-critique pipeline.

    Step 1 (Generate): Produce initial response with generation_model (cheap)
    Step 2 (Critique): Use critique_model to identify specific issues (smart)
    Step 3 (Revise):   Use revision_model to produce improved version (smart)

    Using different models per step is intentional — cheap for first draft,
    smart for critique and revision.
    """
    if verbose:
        print("SELF-CRITIQUE PIPELINE")
        print("=" * 70)

    # Step 1: Generate
    if verbose:
        print(f"
Step 1: Generating initial response ({generation_model})...")
    initial = client.chat.completions.create(
        model=generation_model, temperature=0.3, max_tokens=700,
        messages=[{"role": "user", "content": task}]
    ).choices[0].message.content

    if verbose:
        print("-" * 70)
        print(initial)

    # Step 2: Critique
    if verbose:
        print(f"

Step 2: Critiquing against {len(criteria)} criteria ({critique_model})...")

    criteria_text = "
".join(f"{i+1}. **{c}**" for i, c in enumerate(criteria))
    critique_prompt = f"""Evaluate this response against each criterion strictly.

TASK: {task}

RESPONSE:
{initial}

CRITERIA:
{criteria_text}

For each criterion, output:
CRITERION N: PASS/PARTIAL/FAIL — [one sentence explaining why]

Then:
OVERALL SCORE: X.X (0.0 to 1.0)
TOP 3 IMPROVEMENTS NEEDED:
1. ...
2. ...
3. ...
"""
    critique = client.chat.completions.create(
        model=critique_model, temperature=0.0, max_tokens=800,
        messages=[{"role": "user", "content": critique_prompt}]
    ).choices[0].message.content

    if verbose:
        print("-" * 70)
        print(critique)

    # Extract score
    score_match   = re.search(r'OVERALL SCORE:\s*([0-9.]+)', critique)
    overall_score = float(score_match.group(1)) if score_match else 0.7

    # Step 3: Revise
    if verbose:
        print(f"

Step 3: Generating improved response ({revision_model})...")

    improve_prompt = f"""Rewrite this response to fix ALL identified issues.

ORIGINAL TASK: {task}

CRITIQUE OF INITIAL RESPONSE:
{critique}

Write a COMPLETE improved response. Do NOT reference the critique — just write the better answer.
"""
    improved = client.chat.completions.create(
        model=revision_model, temperature=0.2, max_tokens=900,
        messages=[{"role": "user", "content": improve_prompt}]
    ).choices[0].message.content

    if verbose:
        print("-" * 70)
        print(improved)
        print(f"
Overall critique score: {overall_score:.1f}/1.0")

    return CritiqueResult(
        task=task, initial_output=initial,
        critique=critique, improved_output=improved, overall_score=overall_score
    )


# Demo
result = self_critique_pipeline(
    task="""Explain the difference between an AI Agent and a simple LLM API call.
Include: when to use each, trade-offs, and a short concrete code example.""",
    criteria=[
        "Technical accuracy — definitions are correct and precise",
        "Completeness — covers BOTH what they ARE and WHEN to use each",
        "Code quality — code example is runnable Python",
        "Clarity — a junior developer with 1 year Python can follow",
        "Conciseness — total under 400 words",
        "Actionability — reader can make a decision after reading"
    ]
)
"""))

cells.append(md("""### 8.4 Constitutional AI — Principle-Based Refinement
"""))

cells.append(code("""# ─── 8.4 Constitutional AI Pattern ──────────────────────────────────────────

# A "constitution" — a set of principles every response must satisfy
AI_AGENT_CONSTITUTION = [
    "Accuracy: All technical claims must be verifiable; no speculation as fact",
    "Safety: Never suggest approaches that could cause data loss or security vulnerabilities",
    "Completeness: Cover the full scope of the question; no major aspects omitted",
    "Practicality: Solutions must be implementable with standard Python libraries",
    "Clarity: Code examples must have comments; jargon explained on first use",
    "Neutrality: When comparing tools, present objective trade-offs, not opinions",
]


def constitutional_refinement(
    task: str,
    constitution: list[str],
    max_revisions: int = 2,
    verbose: bool = True
) -> str:
    """
    Iteratively refine a response against a constitution.
    Stops early if all principles pass or max_revisions reached.
    """
    if verbose:
        print(f"CONSTITUTIONAL AI REFINEMENT")
        print(f"Constitution: {len(constitution)} principles | Max revisions: {max_revisions}")
        print("=" * 70)

    # Generate initial draft
    current = client.chat.completions.create(
        model=DEFAULT_MODEL, temperature=0.3, max_tokens=700,
        messages=[{"role": "user", "content": task}]
    ).choices[0].message.content

    for revision in range(1, max_revisions + 1):
        if verbose:
            print(f"
Revision {revision}: Checking against constitution...")

        principles_text = "
".join(f"{i+1}. {p}" for i, p in enumerate(constitution))
        check_prompt = f"""Evaluate this response against each constitutional principle.

RESPONSE:
{current}

CONSTITUTION:
{principles_text}

For each principle output: PRINCIPLE N: PASS/FAIL — [one line why]
Then: VIOLATIONS_FOUND: YES or NO
"""
        check = client.chat.completions.create(
            model=DEFAULT_MODEL, temperature=0.0, max_tokens=600,
            messages=[{"role": "user", "content": check_prompt}]
        ).choices[0].message.content

        if verbose:
            print(check[:400] + ("..." if len(check) > 400 else ""))

        if "VIOLATIONS_FOUND: YES" not in check:
            if verbose:
                print(f"
All principles satisfied after {revision-1} revision(s)!")
            break

        # Revise to fix violations
        revise_prompt = f"""Rewrite this response to fix ALL constitutional violations.

ORIGINAL TASK: {task}
CURRENT RESPONSE:
{current}

VIOLATIONS TO FIX:
{check}

Write a COMPLETE revised response that passes all principles.
"""
        current = client.chat.completions.create(
            model=SMART_MODEL, temperature=0.2, max_tokens=900,
            messages=[{"role": "user", "content": revise_prompt}]
        ).choices[0].message.content

        if verbose:
            print(f"
Revised response (revision {revision}):")
            print("-" * 70)
            print(current[:500] + ("..." if len(current) > 500 else ""))

    return current


# Demo
final_response = constitutional_refinement(
    task="How should I store API keys in my Python AI agent application?",
    constitution=AI_AGENT_CONSTITUTION,
    max_revisions=2
)
print("
FINAL CONSTITUTIONALLY-VALID RESPONSE:")
print("-" * 70)
print(final_response)


### 8.5 Practice Exercises — Advanced Techniques

**Exercise 8A:** Use `meta_prompt()` to generate a system prompt for a *SQL Query Generator* that converts natural language to SQL. Test it on 5 different natural language questions against the AI models database from Exercise 7B.

**Exercise 8B:** Extend `self_critique_pipeline()` to support multiple passes (not just one). Loop until the overall critique score exceeds 0.90 or after 3 passes maximum. Print a summary showing quality improvement per pass.

**Exercise 8C:** Create a custom constitution for a *Code Review Agent* with 8 principles (security, performance, readability, test coverage, documentation, type hints, SOLID principles, error handling). Test it on 3 different Python functions.


---
## 📈 Section 9: Prompt Evaluation & A/B Testing

### 9.1 Theory: Why You Must Measure Prompts

**"If you can't measure it, you can't improve it."** — W. Edwards Deming

Most developers evaluate prompts by eyeballing outputs. This is the #1 cause of prompt regression in production — a prompt that "looks better" subjectively often performs worse on the full distribution of inputs.

#### The Eval Stack

```
Level 1 — Functional Tests  : Does output match expected? (exact match / regex)
Level 2 — LLM-as-Judge      : Does a stronger model rate it as good? (1.0 scale)
Level 3 — Human Evaluation  : Do target users prefer it? (A/B with real traffic)
Level 4 — Business Metrics  : Does it move the metric you care about? (CSAT, conversion)
```

For most teams, **Level 2 (LLM-as-judge)** gives the best signal-to-cost ratio. It:
- Correlates ~0.70–0.85 with human judgment (Zheng et al., 2023)
- Costs ~0.01¢ per evaluation
- Can run in CI/CD pipelines automatically

#### The 5 Metrics Every Prompt Should Track

| Metric | What it measures | Tool |
|--------|-----------------|------|
| Mean Score | Overall quality on eval set | LLM-as-judge |
| Pass Rate | % of cases scoring ≥ threshold | LLM-as-judge |
| Format Compliance | % outputs matching required format | Regex / JSON parse |
| P95 Latency | Response time at 95th percentile | time.time() |
| Cost per call | Input + output tokens × price | tiktoken |

#### Evaluation Design Principles

1. **Separate eval set** — Never evaluate on examples you used to design the prompt (overfitting!)
2. **Diverse coverage** — Test easy, hard, edge, and adversarial cases
3. **Weighted scoring** — Business-critical edge cases should count more
4. **Statistical significance** — Run ≥ 30 samples per prompt before declaring a winner
5. **Multiple metrics** — Accuracy alone is insufficient; always measure latency and cost


In [ ]:
# ─── 9.2 Evaluation Data Structures ─────────────────────────────────────────

@dataclass
class EvalCase:
    """A single evaluation test case."""
    id:              str
    input:           str
    expected_output: str
    weight:          float = 1.0       # Higher = more important to business
    category:        str   = "general" # For group-level analysis
    description:     str   = ""


@dataclass
class EvalResult:
    """Result of evaluating one case."""
    case_id:       str
    actual:        str
    score:         float
    latency_ms:    float
    input_tokens:  int
    output_tokens: int
    pass_threshold: float = 0.7

    @property
    def passed(self) -> bool:
        return self.score >= self.pass_threshold

    @property
    def cost_usd(self) -> float:
        return (self.input_tokens * 0.15 + self.output_tokens * 0.60) / 1_000_000


# Build a comprehensive eval suite for sentiment analysis
SENTIMENT_EVAL_SUITE = [
    # Easy positives
    EvalCase("p001", "This product exceeded my expectations! Fast shipping and great quality.", "POSITIVE", 1.0, "easy_positive"),
    EvalCase("p002", "Absolutely love it. Best purchase I have made this year.", "POSITIVE", 1.0, "easy_positive"),

    # Easy negatives
    EvalCase("n001", "Terrible quality. Broke after one day. Avoid.", "NEGATIVE", 1.0, "easy_negative"),
    EvalCase("n002", "Complete waste of money. The description was totally misleading.", "NEGATIVE", 1.0, "easy_negative"),

    # Easy neutrals
    EvalCase("neu001", "It is okay. Nothing special but does what it says.", "NEUTRAL", 1.0, "easy_neutral"),

    # Edge cases — this is where prompts diverge (weighted higher)
    EvalCase("e001", "Good product but the packaging was damaged.", "NEUTRAL", 2.0, "edge", "Mixed: positive product, negative packaging"),
    EvalCase("e002", "Arrived 2 weeks late, but the item itself is amazing.", "NEUTRAL", 2.0, "edge", "Mixed: negative delivery, positive item"),
    EvalCase("e003", "I guess it works. Nothing to complain about really.", "NEUTRAL", 2.0, "edge", "Lukewarm positive phrased neutrally"),
    EvalCase("e004", "If you like wasting money, this is perfect for you.", "NEGATIVE", 2.5, "edge", "Sarcasm — must detect negative sentiment"),
    EvalCase("e005", "Shockingly... it actually works?!", "POSITIVE", 2.5, "edge", "Surprised positive — unusual phrasing"),

    # Adversarial
    EvalCase("a001", "POSITIVE reviews are fake. This is NEGATIVE all the way.", "NEGATIVE", 3.0, "adversarial", "Contains the word POSITIVE but is negative"),
    EvalCase("a002", "Rating: 1/5 — DO NOT BUY.", "NEGATIVE", 3.0, "adversarial", "Numeric rating format, no prose"),
]

print(f"Eval suite: {len(SENTIMENT_EVAL_SUITE)} cases")
cats = Counter(c.category for c in SENTIMENT_EVAL_SUITE)
for cat, count in sorted(cats.items()):
    print(f"  {cat:<15}: {count} cases")


In [ ]:
# ─── 9.3 LLM-as-Judge Evaluator ──────────────────────────────────────────────

class PromptEvaluator:
    """
    Production-grade prompt evaluator using LLM-as-judge.
    Supports weighted scoring, category analysis, cost/latency tracking.
    """

    JUDGE_SYSTEM = """You are an impartial evaluation judge for NLP task outputs.
Score how well 'actual_output' answers the task given 'expected_output'.
Be strict and objective. Return ONLY valid JSON."""

    def __init__(self, eval_cases: list[EvalCase], pass_threshold: float = 0.7):
        self.cases          = eval_cases
        self.pass_threshold = pass_threshold

    def score_single(self, actual: str, expected: str, context: str = "") -> tuple[float, str]:
        """Use LLM-as-judge to score one prediction. Returns (score, reasoning)."""
        judge_prompt = f"""Score how well the actual output matches the expected output.

Expected: {expected}
Actual:   {actual}
Context:  {context or 'General classification task'}

Scoring rubric:
1.0 = Perfect match in meaning and format
0.8 = Correct meaning, minor format difference
0.5 = Partially correct
0.2 = Wrong but related
0.0 = Completely wrong

Return JSON: {{"score": 0.0_to_1.0, "reason": "one sentence"}}"""

        try:
            resp = client.chat.completions.create(
                model=DEFAULT_MODEL, temperature=0.0, max_tokens=100,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": self.JUDGE_SYSTEM},
                    {"role": "user",   "content": judge_prompt}
                ]
            )
            parsed = json.loads(resp.choices[0].message.content)
            return float(parsed.get("score", 0.0)), parsed.get("reason", "")
        except Exception as e:
            return 0.0, f"Error: {e}"

    def evaluate_prompt(self, system_prompt: str, prompt_name: str = "prompt") -> dict:
        """Evaluate a system prompt across all test cases. Returns comprehensive stats."""
        results: list[EvalResult] = []

        for case in self.cases:
            t0 = time.time()
            try:
                resp = client.chat.completions.create(
                    model=DEFAULT_MODEL, temperature=0.1, max_tokens=100,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user",   "content": case.input}
                    ]
                )
                latency_ms  = (time.time() - t0) * 1000
                actual      = resp.choices[0].message.content.strip()
                score, _    = self.score_single(actual, case.expected_output, case.description)
                results.append(EvalResult(
                    case_id=case.id, actual=actual,
                    score=score * case.weight,
                    latency_ms=latency_ms,
                    input_tokens=resp.usage.prompt_tokens,
                    output_tokens=resp.usage.completion_tokens,
                    pass_threshold=self.pass_threshold
                ))
            except Exception as e:
                print(f"  Case {case.id} failed: {e}")

        if not results:
            return {"error": "No results"}

        scores    = [r.score for r in results]
        latencies = [r.latency_ms for r in results]
        total_cost = sum(r.cost_usd for r in results)

        cat_scores: dict[str, list] = defaultdict(list)
        for r, c in zip(results, self.cases):
            cat_scores[c.category].append(r.score)

        return {
            "prompt_name":    prompt_name,
            "n_cases":        len(results),
            "mean_score":     round(statistics.mean(scores), 3),
            "median_score":   round(statistics.median(scores), 3),
            "min_score":      round(min(scores), 3),
            "pass_rate":      round(sum(r.passed for r in results) / len(results), 3),
            "avg_latency_ms": round(statistics.mean(latencies), 1),
            "p95_latency_ms": round(sorted(latencies)[int(len(latencies)*0.95)], 1),
            "total_cost_usd": round(total_cost, 5),
            "cost_per_case":  round(total_cost / len(results), 6),
            "by_category":    {cat: round(statistics.mean(sc), 3) for cat, sc in cat_scores.items()},
        }


evaluator = PromptEvaluator(SENTIMENT_EVAL_SUITE)
print("✅ PromptEvaluator ready!")


In [ ]:
# ─── 9.4 A/B/C Testing — Compare 3 Prompt Versions ─────────────────────────

# Prompt A: Minimal (baseline)
PROMPT_A = "Classify the sentiment of the customer review as POSITIVE, NEGATIVE, or NEUTRAL."

# Prompt B: Structured with rules
PROMPT_B = """You are a sentiment analysis expert for e-commerce platforms.

Classify customer reviews as exactly one of: POSITIVE | NEGATIVE | NEUTRAL

Definitions:
- POSITIVE: Customer is overall satisfied, would recommend
- NEGATIVE: Customer is dissatisfied, would not recommend
- NEUTRAL: Mixed feelings — both positive and negative aspects

Edge case rules:
- Sarcasm (e.g., "oh great, another broken item") → NEGATIVE
- Surprised positive ("shockingly it works!") → POSITIVE
- Numeric rating only: 4-5 stars → POSITIVE, 1-2 stars → NEGATIVE
- Contains the word POSITIVE/NEGATIVE but opposite meaning → follow actual meaning

Output: Only the classification label. Nothing else."""

# Prompt C: Few-shot examples
PROMPT_C = """Sentiment analysis expert. Classify as POSITIVE, NEGATIVE, or NEUTRAL.

Examples:
Review: "Works perfectly, shipping was fast" → POSITIVE
Review: "Broke after 2 days, terrible quality" → NEGATIVE
Review: "Decent quality but a bit overpriced" → NEUTRAL
Review: "Oh sure, waterproof they said. Ruined in 5 minutes." → NEGATIVE [sarcasm]
Review: "I was skeptical but this actually works!" → POSITIVE [surprised positive]

Output ONLY the label."""

# Run evaluations
print("A/B/C Test: Sentiment Analysis Prompts")
print(f"Evaluating {len(SENTIMENT_EVAL_SUITE)} test cases per prompt...")
results = {}
for name, prompt in [("A_minimal", PROMPT_A), ("B_structured", PROMPT_B), ("C_fewshot", PROMPT_C)]:
    print(f"
Evaluating Prompt {name}...")
    results[name] = evaluator.evaluate_prompt(prompt, prompt_name=name)

# Print comparison table
print("
" + "=" * 80)
print("A/B/C TEST RESULTS")
print("=" * 80)
metrics = ["mean_score", "pass_rate", "min_score", "avg_latency_ms", "p95_latency_ms", "cost_per_case"]
print(f"{'Metric':<22} {'A (Minimal)':>14} {'B (Structured)':>16} {'C (Few-shot)':>14} {'Winner':>8}")
print("-" * 80)
for metric in metrics:
    vals = {k: results[k][metric] for k in results}
    is_lower_better = metric in ["avg_latency_ms", "p95_latency_ms", "cost_per_case"]
    best_key = min(vals, key=vals.get) if is_lower_better else max(vals, key=vals.get)
    winner = best_key.split("_")[0]
    print(f"{metric:<22} {str(vals['A_minimal']):>14} {str(vals['B_structured']):>16} {str(vals['C_fewshot']):>14} {winner:>8} ✅")

print("
CATEGORY BREAKDOWN:")
print(f"{'Category':<20} {'A':>10} {'B':>10} {'C':>10}")
print("-" * 55)
cats = set()
for r in results.values():
    cats.update(r["by_category"].keys())
for cat in sorted(cats):
    a = results["A_minimal"]["by_category"].get(cat, "—")
    b = results["B_structured"]["by_category"].get(cat, "—")
    c = results["C_fewshot"]["by_category"].get(cat, "—")
    print(f"{cat:<20} {str(a):>10} {str(b):>10} {str(c):>10}")


### 9.5 Regression Testing — CI/CD for Prompts

In production, every prompt change should pass automated regression testing before deployment.


In [ ]:
# ─── 9.5 Prompt Regression Suite ────────────────────────────────────────────

class PromptRegressionSuite:
    """
    Automated regression testing for prompts. Use in CI/CD to catch prompt
    degradations before they reach production.

    Typical workflow:
      1. Developer proposes prompt change
      2. CI runs regression suite
      3. If mean_score drops > 3% or pass_rate drops > 5% -> BLOCK deploy
      4. If metrics improve -> ALLOW deploy with change log entry
    """

    def __init__(
        self,
        baseline_prompt: str,
        eval_cases:      list[EvalCase],
        thresholds:      dict = None
    ):
        self.baseline_prompt = baseline_prompt
        self.evaluator       = PromptEvaluator(eval_cases)
        self.baseline_result = None
        self.thresholds = thresholds or {
            "mean_score_min":    0.70,
            "pass_rate_min":     0.65,
            "score_regression":  0.03,   # Max 3% score drop allowed
        }

    def establish_baseline(self) -> dict:
        print("Establishing baseline...")
        self.baseline_result = self.evaluator.evaluate_prompt(self.baseline_prompt, "baseline")
        print(f"  Baseline mean_score: {self.baseline_result['mean_score']}")
        print(f"  Baseline pass_rate:  {self.baseline_result['pass_rate']}")
        return self.baseline_result

    def test_candidate(self, candidate_prompt: str, name: str = "candidate") -> dict:
        """Test a candidate prompt against baseline. Returns approval decision."""
        if not self.baseline_result:
            self.establish_baseline()

        print(f"
Testing candidate: '{name}'")
        cand = self.evaluator.evaluate_prompt(candidate_prompt, name)
        b, c = self.baseline_result, cand

        checks = {
            "quality_floor": {
                "pass": c["mean_score"] >= self.thresholds["mean_score_min"],
                "baseline": b["mean_score"], "candidate": c["mean_score"],
                "threshold": self.thresholds["mean_score_min"]
            },
            "no_regression": {
                "pass": (c["mean_score"] - b["mean_score"]) >= -self.thresholds["score_regression"],
                "baseline": b["mean_score"], "candidate": c["mean_score"],
                "delta": round(c["mean_score"] - b["mean_score"], 4)
            },
            "pass_rate": {
                "pass": c["pass_rate"] >= self.thresholds["pass_rate_min"],
                "baseline": b["pass_rate"], "candidate": c["pass_rate"]
            },
        }

        all_passed = all(v["pass"] for v in checks.values())

        print(f"
{'Check':<25} {'Baseline':>10} {'Candidate':>10} {'Status':>10}")
        print("-" * 60)
        for check_name, data in checks.items():
            status = "PASS" if data["pass"] else "FAIL"
            print(f"{check_name:<25} {str(data['baseline']):>10} {str(data['candidate']):>10} {status:>10}")

        verdict = "APPROVED — Safe to deploy" if all_passed else "BLOCKED — Do not deploy"
        print(f"
VERDICT: {verdict}")

        return {"approved": all_passed, "checks": checks, "candidate_result": cand}


# Demo: Test Prompt C as a candidate to replace Prompt A
suite = PromptRegressionSuite(PROMPT_A, SENTIMENT_EVAL_SUITE)
suite.establish_baseline()
decision = suite.test_candidate(PROMPT_C, name="C_fewshot_candidate")


---
## 🏭 Section 10: Production — Smart Prompt Management System

### 10.1 Theory: What Changes in Production

| Dimension | Dev/Notebook | Production |
|-----------|-------------|------------|
| Load | 1 request | 100–10,000 req/min |
| Errors | Crash is acceptable | Must recover gracefully |
| Cost | "I'll pay it" | $0.001/call × millions |
| Latency | 2–5s fine | P95 < 1s required |
| Versions | One prompt | Multiple concurrent versions, A/B |
| Audit | Not needed | Full trace for compliance |
| Caching | Never considered | 30–60% of traffic cacheable |
| Observability | print() | Structured logs + metrics + alerts |

#### The Production Prompt Lifecycle

```
Design → Eval → Canary Deploy (5%) → Full Deploy → Monitor → Deprecate → Retire
    ↑_______________________________________________|
                     (regression → rollback)
```

#### Key Production Patterns

1. **Semantic Caching** — MD5 hash of prompt + inputs; if seen before, return cached result. Saves 40–60% of API costs.
2. **Version Control** — Treat prompts like code: semantic versioning, changelogs, rollback.
3. **Rate Limiting** — Prevent a single tenant from consuming all capacity (semaphore-based).
4. **Circuit Breaker** — If error rate > threshold, stop calling the LLM and use cached/fallback responses.
5. **Async Batching** — Group concurrent requests; lower cost, same effective latency.


In [ ]:
# ─── 10.2 Production Prompt Manager ─────────────────────────────────────────

class PromptVersion(BaseModel):
    """Version-controlled prompt with full metadata."""
    name:            str
    version:         str
    system_template: str
    user_template:   str
    model:           str = DEFAULT_MODEL
    max_tokens:      int = 1000
    temperature:     float = 0.1
    variables:       list[str] = Field(default_factory=list)
    description:     str = ""
    author:          str = "unknown"
    created_at:      str = Field(default_factory=lambda: datetime.now().isoformat())
    deprecated:      bool = False
    changelog:       str = ""


class UsageStats(BaseModel):
    total_calls:     int   = 0
    cache_hits:      int   = 0
    total_tokens:    int   = 0
    total_cost_usd:  float = 0.0
    error_count:     int   = 0
    latency_samples: list  = Field(default_factory=list)

    def record_latency(self, ms: float):
        self.latency_samples.append(ms)

    @property
    def p50_latency_ms(self) -> float:
        if not self.latency_samples: return 0.0
        s = sorted(self.latency_samples)
        return s[len(s)//2]

    @property
    def p95_latency_ms(self) -> float:
        if not self.latency_samples: return 0.0
        s = sorted(self.latency_samples)
        return s[int(len(s)*0.95)]


class SmartPromptManager:
    """
    Production-grade prompt management with:
    - Version control and deprecation
    - Semantic response caching with TTL
    - Token counting and cost tracking per version
    - Latency percentiles (P50, P95)
    - Error rate monitoring
    - Template variable validation
    """

    PRICING = {
        "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }

    def __init__(self, cache_ttl_seconds: int = 3600):
        self._prompts: dict[str, PromptVersion] = {}
        self._cache:   dict[str, tuple] = {}   # key → (result, timestamp)
        self._stats:   dict[str, UsageStats] = {}
        self.cache_ttl = cache_ttl_seconds
        self._call_log: list[dict] = []

    def register(self, prompt: PromptVersion) -> None:
        key = f"{prompt.name}:{prompt.version}"
        if prompt.deprecated:
            print(f"Warning: Registering deprecated prompt {key}")
        self._prompts[key] = prompt
        self._stats[key]   = UsageStats()
        print(f"Registered: {key} — {prompt.description}")

    def deprecate(self, name: str, version: str) -> None:
        key = f"{name}:{version}"
        if key in self._prompts:
            self._prompts[key].deprecated = True
            print(f"Deprecated: {key}")

    def list_versions(self, name: str) -> list[str]:
        return [k for k in self._prompts if k.startswith(f"{name}:")]

    def _render(self, template: str, **kwargs) -> str:
        for k, v in kwargs.items():
            template = template.replace(f"{{{k}}}", str(v))
        remaining = re.findall(r'\{(\w+)\}', template)
        if remaining:
            print(f"Warning: Unfilled template variables: {remaining}")
        return template

    def _validate_variables(self, prompt: PromptVersion, kwargs: dict) -> None:
        missing = [v for v in prompt.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required variables for {prompt.name}: {missing}")

    def _cache_key(self, prompt_key: str, **kwargs) -> str:
        content = f"{prompt_key}:{json.dumps(kwargs, sort_keys=True)}"
        return hashlib.md5(content.encode()).hexdigest()

    def _cache_get(self, key: str) -> Optional[str]:
        if key in self._cache:
            result, timestamp = self._cache[key]
            if time.time() - timestamp < self.cache_ttl:
                return result
            del self._cache[key]
        return None

    def _cache_set(self, key: str, value: str) -> None:
        self._cache[key] = (value, time.time())

    def _calculate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        if model not in self.PRICING: return 0.0
        p = self.PRICING[model]
        return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

    def run(
        self,
        prompt_name: str,
        version:     str = "latest",
        use_cache:   bool = True,
        **kwargs
    ) -> str:
        """Execute a registered prompt with full observability."""
        if version == "latest":
            versions = self.list_versions(prompt_name)
            if not versions:
                raise ValueError(f"No prompt named '{prompt_name}' registered")
            version = sorted(versions)[-1].split(":")[-1]

        key = f"{prompt_name}:{version}"
        if key not in self._prompts:
            raise ValueError(f"Prompt '{key}' not found. Available: {list(self._prompts.keys())}")

        pv, stats = self._prompts[key], self._stats[key]
        stats.total_calls += 1

        if pv.deprecated:
            print(f"Warning: Using deprecated prompt {key}. Please upgrade.")

        self._validate_variables(pv, kwargs)

        cache_key = self._cache_key(key, **kwargs)
        cached    = self._cache_get(cache_key) if use_cache else None
        if cached:
            stats.cache_hits += 1
            print(f"Cache hit! (saved ~${self._calculate_cost(pv.model, 500, 200):.6f})")
            return cached

        system = self._render(pv.system_template, **kwargs)
        user   = self._render(pv.user_template,   **kwargs)

        t0 = time.time()
        try:
            response = client.chat.completions.create(
                model=pv.model, temperature=pv.temperature, max_tokens=pv.max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user}
                ]
            )
        except Exception as e:
            stats.error_count += 1
            raise

        latency_ms   = (time.time() - t0) * 1000
        result       = response.choices[0].message.content
        cost         = self._calculate_cost(pv.model, response.usage.prompt_tokens, response.usage.completion_tokens)

        stats.total_tokens   += response.usage.total_tokens
        stats.total_cost_usd += cost
        stats.record_latency(latency_ms)

        if use_cache:
            self._cache_set(cache_key, result)

        self._call_log.append({
            "timestamp": datetime.now().isoformat(), "prompt": key,
            "latency_ms": round(latency_ms, 1), "cost_usd": round(cost, 6)
        })

        print(f"{key} | ${cost:.6f} | {response.usage.total_tokens} tokens | {latency_ms:.0f}ms")
        return result

    def get_stats(self, prompt_name: str, version: str = "latest") -> dict:
        if version == "latest":
            versions = self.list_versions(prompt_name)
            version  = sorted(versions)[-1].split(":")[-1]
        key   = f"{prompt_name}:{version}"
        stats = self._stats.get(key, UsageStats())
        cache_rate = stats.cache_hits / stats.total_calls if stats.total_calls else 0
        return {
            "prompt":       key,
            "total_calls":  stats.total_calls,
            "cache_rate":   f"{cache_rate:.0%}",
            "error_count":  stats.error_count,
            "total_cost":   f"${stats.total_cost_usd:.6f}",
            "p50_latency":  f"{stats.p50_latency_ms:.0f}ms",
            "p95_latency":  f"{stats.p95_latency_ms:.0f}ms",
        }

    def print_dashboard(self) -> None:
        print("
" + "=" * 80)
        print("PROMPT MANAGER DASHBOARD")
        print("=" * 80)
        print(f"{'Prompt Version':<30} {'Calls':>8} {'Cache%':>8} {'AvgCost$':>10} {'P95ms':>8} {'Errors':>8}")
        print("-" * 80)
        for key, stats in self._stats.items():
            if stats.total_calls == 0: continue
            cache_rate = stats.cache_hits / stats.total_calls
            avg_cost   = stats.total_cost_usd / stats.total_calls
            depr       = " [DEPRECATED]" if self._prompts[key].deprecated else ""
            print(f"{key+depr:<30} {stats.total_calls:>8,} {cache_rate:>8.0%} {avg_cost:>10.6f} {stats.p95_latency_ms:>8.0f} {stats.error_count:>8}")
        print("=" * 80)
        print(f"Cache size: {len(self._cache)} entries | Total logged calls: {len(self._call_log)}")


print("✅ SmartPromptManager defined!")


In [ ]:
# ─── 10.3 Register & Demo the Production Manager ─────────────────────────────

manager = SmartPromptManager(cache_ttl_seconds=300)

# Register v1
manager.register(PromptVersion(
    name="sentiment",
    version="v1.0",
    description="Original minimal sentiment classifier",
    author="alice@team.com",
    system_template="Classify the sentiment of text about {domain} as POSITIVE, NEGATIVE, or NEUTRAL. Output one word.",
    user_template="Text: {text}",
    model=DEFAULT_MODEL, temperature=0.0,
    variables=["domain", "text"],
    changelog="Initial version"
))

# Register v1.1 with improvements
manager.register(PromptVersion(
    name="sentiment",
    version="v1.1",
    description="Added edge case handling and sarcasm detection",
    author="bob@team.com",
    system_template="""You are a sentiment expert for {domain}.
Classify as POSITIVE, NEGATIVE, or NEUTRAL.
Sarcasm (e.g., "oh great, another failure") → NEGATIVE
Surprised positive ("shockingly it works!") → POSITIVE
Output ONLY the label.""",
    user_template="Text: {text}",
    model=DEFAULT_MODEL, temperature=0.0,
    variables=["domain", "text"],
    changelog="Added sarcasm detection. +8% accuracy on edge cases."
))

# Deprecate v1.0
manager.deprecate("sentiment", "v1.0")

# Run demo requests
reviews = [
    ("e-commerce", "Great product, fast shipping, will buy again!"),
    ("e-commerce", "Arrived broken, terrible customer service."),
    ("e-commerce", "Oh great, another premium product that lasts 2 days."),  # Sarcasm
    ("tech support", "The software crashes every time I try to export."),
    ("e-commerce", "Great product, fast shipping, will buy again!"),  # Cache hit!
]

print("
Running SmartPromptManager demo...
")
for domain, text in reviews:
    print(f"
Input: {text[:60]}...")
    result = manager.run("sentiment", version="v1.1", domain=domain, text=text)
    print(f"Output: {result}")

manager.print_dashboard()


In [ ]:
# ─── 10.4 Async Batch Processing ─────────────────────────────────────────────

async def batch_classify_async(
    reviews: list[str],
    system_prompt: str,
    model: str = DEFAULT_MODEL,
    max_concurrent: int = 5
) -> list[dict]:
    """
    Classify multiple reviews concurrently using asyncio.
    Speedup: ~4–8x faster than sequential for typical I/O-bound workloads.
    Respects rate limits via semaphore (max_concurrent).
    """
    semaphore = asyncio.Semaphore(max_concurrent)

    async def classify_one(review: str, idx: int) -> dict:
        async with semaphore:
            t0 = time.time()
            try:
                response = await async_client.chat.completions.create(
                    model=model, temperature=0.0, max_tokens=10,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user",   "content": review}
                    ]
                )
                return {
                    "idx": idx,
                    "review": review[:50],
                    "result": response.choices[0].message.content.strip(),
                    "latency_ms": round((time.time() - t0) * 1000, 1),
                    "tokens": response.usage.total_tokens,
                    "error": None
                }
            except Exception as e:
                return {"idx": idx, "review": review[:50], "result": "ERROR", "error": str(e)}

    tasks = [classify_one(r, i) for i, r in enumerate(reviews)]
    return await asyncio.gather(*tasks)


# Benchmark async vs what sequential would take
TEST_REVIEWS = [
    "Love this product! Works exactly as described.",
    "Complete garbage. Save your money.",
    "It is okay, nothing special.",
    "Absolute perfection. Five stars.",
    "Terrible. Broke on day one.",
    "Decent quality but a bit pricey.",
    "Outstanding! Far exceeded expectations.",
    "Not what I expected. Disappointed.",
]

simple_prompt = "Classify: POSITIVE, NEGATIVE, or NEUTRAL. One word only."

print(f"Async Batch: Processing {len(TEST_REVIEWS)} reviews (max 5 concurrent)...")
t0      = time.time()
results = await batch_classify_async(TEST_REVIEWS, simple_prompt)
elapsed = time.time() - t0

print(f"
Completed in {elapsed:.2f}s (avg {elapsed/len(TEST_REVIEWS)*1000:.0f}ms/review)")
print(f"
{'#':>3} {'Result':<12} {'Latency':>10}  Preview")
print("-" * 55)
for r in results:
    print(f"{r['idx']+1:>3} {r['result']:<12} {r['latency_ms']:>8.0f}ms  {r['review']}")
print(f"
Total tokens: {sum(r.get('tokens',0) for r in results):,}")


### 10.5 Practice Exercises — Production Systems

**Exercise 10A:** Add a **Circuit Breaker** to `SmartPromptManager`. When `error_count / total_calls > 0.10`, automatically switch to a fallback model and log an alert. Resume normal operation when error rate drops below 5%.

**Exercise 10B:** Implement **prompt version rollback** in `SmartPromptManager`. Add a `rollback(name, from_version, to_version)` method that deprecates the current version, marks the target version as active, and logs the event with timestamp and reason.

**Exercise 10C:** Build a **cost budget enforcer** — if `total_cost_usd > budget_limit`, automatically switch to the cheapest model and notify via a configurable callback.


---
## 🚫 Section 11: Anti-Patterns — What NOT to Do

These are the 8 most common mistakes that kill prompt quality in production. Each has been observed in real AI systems.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║               TOP 8 PROMPT ENGINEERING ANTI-PATTERNS                    ║
╚══════════════════════════════════════════════════════════════════════════╝

ANTI-PATTERN 1 — Vague Role Definition
BAD:  "You are a helpful assistant."
GOOD: "You are a senior Python security engineer specializing in OWASP Top 10
       vulnerabilities for enterprise fintech applications."
WHY:  Specific roles with stakes activate domain knowledge and reduce hedging.

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 2 — Negative-Only Instructions
BAD:  "Don't be verbose. Don't use jargon. Don't give opinions."
GOOD: "Reply in 3 bullet points max. Use plain English for a non-technical
       manager. State facts only, no editorializing."
WHY:  Models violate negative instructions ~25-35% of the time (Shah et al.).

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 3 — The Monolith Prompt
BAD:  "Summarize, translate to French, check for bias, format as JSON, email result."
GOOD: Separate prompt chains: Summarize → Translate → Check → Format
WHY:  Tasks at the end of long lists get ~40% lower quality (lost-in-the-middle).

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 4 — No Output Constraints
BAD:  "Analyze this customer support ticket."
GOOD: "Analyze this ticket. Return JSON: {priority: 1-5, category: str, summary: max 20 words}"
WHY:  Unconstrained output wastes tokens and breaks downstream parsing.

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 5 — Inconsistent Few-Shot Examples
BAD:  Example 1: JSON | Example 2: markdown | Example 3: plain text
GOOD: All examples use identical structure and format
WHY:  Format inconsistency creates ~2x higher output format error rate.

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 6 — Wrong Temperature for the Task
BAD:  temperature=0 for brainstorming / creative tasks
BAD:  temperature=1.0 for JSON output / deterministic classification
GOOD: 0.0-0.2 for deterministic tasks | 0.7-1.0 for creative tasks
WHY:  temp=0 for creative tasks produces repetitive, "canonical" outputs.

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 7 — Ignoring Model-Specific Quirks
GPT-4o:     Use function calling, not text-based ReAct for tools
Claude:     XML tags work better than markdown headers
Llama:      System prompt in user turn often outperforms system field
Gemini:     "Think step by step" in the user turn, not system prompt
WHY:  Same prompt can perform 30-40% differently across models.

══════════════════════════════════════════════════════════════════════════
ANTI-PATTERN 8 — No Evaluation Before Deploy
BAD:  "Looks good in my 5-case test → ship it"
GOOD: Eval suite (30+ cases) → regression guard → canary 5% → monitor → 100%
WHY:  30% of prompt regressions are invisible to manual spot-checking.
""")


In [ ]:
# ─── Live Demo: Anti-Pattern #3 (Monolith) vs Chaining ─────────────────────

ticket = """I have been waiting 3 WEEKS for my refund. I emailed 5 times with
no response. This is unacceptable. I am about to dispute with my bank."""

# Approach 1: Monolith (Anti-pattern #3)
monolith = """Analyze this support ticket, classify by urgency (1-5), identify
customer emotion, suggest response approach, estimate resolution time, format as JSON.

Ticket: """ + ticket

print("MONOLITH APPROACH:")
print("-" * 60)
_ = chat(monolith, max_tokens=600)

print("

CHAINED APPROACH (3 specialized prompts):")
print("-" * 60)

# Step 1: Classify
step1 = chat(
    ticket,
    system_message='Classify this support ticket. JSON only: {"urgency": 1-5, "category": "refund|technical|general", "emotion": "angry|frustrated|confused"}',
    verbose=False
)
print(f"Step 1 (Classify): {step1}")

# Step 2: Strategy (uses Step 1 output)
step2 = chat(
    f"Ticket: {ticket}
Classification: {step1}",
    system_message='Based on this classified ticket, output JSON: {"tone": str, "key_actions": [str], "resolution_hours": int, "escalate": bool}',
    verbose=False
)
print(f"Step 2 (Strategy): {step2}")

# Step 3: Draft response (uses both outputs)
print("Step 3 (Draft response):")
_ = chat(
    f"Ticket: {ticket}
Classification: {step1}
Strategy: {step2}",
    system_message="Write a professional customer support response following the strategy. Max 80 words. Acknowledge the wait time in the first sentence.",
    max_tokens=200
)


---
## 🏆 Section 12: Capstone Project — Production AI Code Review Agent

This capstone combines **every technique from Part 1 and Part 2** into a single production-ready system:

| Component | Technique Used |
|-----------|---------------|
| Agent loop | ReAct Pattern (Section 7) |
| Prompt generation | Meta-Prompting (Section 8.2) |
| Quality improvement | Self-Critique (Section 8.3) |
| Quality measurement | LLM-as-judge Evaluation (Section 9) |
| State management | SmartPromptManager (Section 10) |
| High-throughput | Async Processing (Section 10.4) |


In [ ]:
# ─── Security & Analysis Tools ───────────────────────────────────────────────

def check_security(code: str) -> str:
    """Scan for common security vulnerabilities using regex patterns."""
    issues = []
    if re.search(r'execute.*f["']|f["'].*\{.*\}.*execute', code, re.IGNORECASE):
        issues.append("SQL_INJECTION: f-string used in SQL execute()")
    if re.search(r'eval\(|exec\(', code):
        issues.append("CODE_INJECTION: eval() or exec() detected")
    if re.search(r'(password|secret|api_key|token)\s*=\s*["'][^"']{4,}', code, re.IGNORECASE):
        issues.append("HARDCODED_SECRET: Hardcoded credential detected")
    if re.search(r'md5|sha1', code, re.IGNORECASE):
        issues.append("WEAK_CRYPTO: MD5/SHA1 are cryptographically broken")
    if re.search(r'subprocess.*shell\s*=\s*True', code):
        issues.append("COMMAND_INJECTION: subprocess with shell=True")
    return f"Security issues: {issues}" if issues else "No obvious security vulnerabilities detected."


def check_code_style(code: str) -> str:
    """Check Python style and best practices."""
    issues = []
    lines = code.split("
")
    long_lines = [i+1 for i, l in enumerate(lines) if len(l) > 120]
    if long_lines: issues.append(f"Lines too long (>120 chars): {long_lines[:3]}")
    if "except:" in code: issues.append("Bare except clause — use 'except Exception as e'")
    if re.search(r'def \w+\([^)]*\):\s*
(?!\s+["'])', code):
        issues.append("Missing docstrings in function definitions")
    return f"Style issues: {issues}" if issues else "Code style looks clean."


def estimate_complexity(code: str) -> str:
    """Estimate cyclomatic complexity."""
    branches   = len(re.findall(r'(if|elif|for|while|and|or|except)', code))
    functions  = len(re.findall(r'^\s*def ', code, re.MULTILINE))
    complexity = branches + 1
    rating = "Low" if complexity < 5 else ("Medium" if complexity < 10 else "High")
    return f"Cyclomatic complexity: {complexity} ({rating}) | Functions: {functions}"


# Register tools for the code review agent
CODE_REVIEW_TOOLS = {
    "check_security":   check_security,
    "check_style":      check_code_style,
    "check_complexity": estimate_complexity,
    "search":           search_web,
    "calculate":        calculate,
}

CODE_REVIEW_SYSTEM = """You are a senior Python code reviewer. Perform a comprehensive review using available tools.

Available tools:
- check_security(code: str)    → Scan for security vulnerabilities
- check_style(code: str)       → Check Python style and best practices
- check_complexity(code: str)  → Estimate cyclomatic complexity
- search(query: str)           → Look up best practices or CVE info
- calculate(expr: str)         → Compute any metrics

REVIEW PROCESS:
1. Run check_security → analyze results
2. Run check_style    → analyze results
3. Run check_complexity → analyze results
4. Search for any specific vulnerability details if needed
5. Final Answer: Structured JSON report:
   {"overall_score": 0-10, "critical_issues": [...], "warnings": [...], "suggestions": [...], "verdict": "approve|request_changes|reject"}

Thought: ...
Action: tool_name("params")
...
Final Answer: {...}
"""


def run_code_review_agent(code: str, filename: str = "code.py") -> dict:
    """Run the full code review agent using the ReAct loop."""
    print(f"
Code Review Agent: {filename}")
    print("=" * 70)

    messages = [
        {"role": "system", "content": CODE_REVIEW_SYSTEM},
        {"role": "user",   "content": f"Review this Python code from {filename}:

{code}"}
    ]

    for step in range(1, 8):
        response = client.chat.completions.create(
            model=SMART_MODEL, temperature=0.1, max_tokens=700, messages=messages
        )
        output = response.choices[0].message.content
        print(f"
[Step {step}]")
        print(output[:400] + ("..." if len(output) > 400 else ""))

        if "Final Answer:" in output:
            final = output.split("Final Answer:")[-1].strip()
            try:
                json_match = re.search(r'\{.*\}', final, re.DOTALL)
                if json_match:
                    report = json.loads(json_match.group())
                    return {"raw": final, "report": report, "steps": step}
            except:
                pass
            return {"raw": final, "report": {}, "steps": step}

        tool_name, tool_input = parse_react_action(output)
        if tool_name and tool_name in CODE_REVIEW_TOOLS:
            observation = CODE_REVIEW_TOOLS[tool_name](tool_input)
            print(f"
Observation: {observation}")
            messages.append({"role": "assistant", "content": output})
            messages.append({"role": "user", "content": f"Observation: {observation}

Continue:"})
        else:
            break

    return {"raw": "Max steps reached", "report": {}, "steps": 7}


# Test on a vulnerable code file
VULNERABLE_CODE = """
import sqlite3
import hashlib
import subprocess

DB_PASSWORD = "admin123_secret"   # Hardcoded credential!

def get_user_data(username, password):
    conn = sqlite3.connect("app.db")
    query = f"SELECT * FROM users WHERE username = '{username}'"  # SQL injection!
    return conn.execute(query).fetchall()

def run_report(report_name):
    result = subprocess.run(f"python reports/{report_name}.py", shell=True, capture_output=True)
    return result.stdout

def hash_password(pwd):
    return hashlib.md5(pwd.encode()).hexdigest()   # Weak hash!

def process_user_input(data):
    result = eval(data)   # Arbitrary code execution!
    return result
"""

review = run_code_review_agent(VULNERABLE_CODE, "auth_service.py")
if review.get("report"):
    print("

STRUCTURED REVIEW REPORT:")
    print(json.dumps(review["report"], indent=2))


---
## 📋 Summary & Key Takeaways

### ✅ Techniques Mastered in Part 2

| Section | Technique | Status |
|---------|-----------|--------|
| 7 | ReAct Pattern — Tool-augmented reasoning | ✅ |
| 7 | Robust action parsing with multi-format fallback | ✅ |
| 7 | Agent retry logic and model fallback | ✅ |
| 8 | Meta-Prompting — LLM-generated optimal prompts | ✅ |
| 8 | Self-Critique 3-step pipeline | ✅ |
| 8 | Constitutional AI — principle-based refinement | ✅ |
| 9 | LLM-as-judge evaluation framework | ✅ |
| 9 | A/B/C testing with statistical comparison | ✅ |
| 9 | CI/CD regression testing for prompts | ✅ |
| 10 | SmartPromptManager with versioning | ✅ |
| 10 | Semantic response caching with TTL | ✅ |
| 10 | Async batch processing with rate limiting | ✅ |
| 10 | Cost tracking and latency percentiles | ✅ |
| 11 | 8 critical anti-patterns with fixes | ✅ |
| 12 | End-to-end AI Code Review Agent capstone | ✅ |

---

### 🗺️ Next Steps in the Roadmap

```
You have completed Phase 1: Prompt Engineering (Parts 1 & 2)

Phase 2 (next):
├── Tool Use — OpenAI Function Calling (structured ReAct, parallel tools)
├── Retrieval-Augmented Generation (RAG)
│   ├── Document loading and chunking strategies
│   ├── Embedding models (text-embedding-3-small/large)
│   ├── Vector databases (Chroma, Pinecone, Weaviate)
│   └── Hybrid search (dense + sparse BM25)
└── Production RAG patterns (re-ranking, HyDE, FLARE)

Phase 3:
├── LangChain — chains, agents, LCEL, callbacks
├── LangGraph — stateful multi-agent workflows with checkpointing
└── CrewAI — role-based multi-agent teams

Phase 4:
├── Multi-agent coordination and communication
├── Long-horizon planning (hierarchical agents)
└── Human-in-the-loop patterns
```

---

### 💡 15 Principles for Production Prompt Engineering

1. **Measure first** — No prompt is "better" without an eval suite proving it
2. **Specificity > Generality** — Specific roles, tasks, and constraints always win
3. **Positive > Negative** — Tell the model what TO do, not what to avoid
4. **Chain > Monolith** — Break complex tasks into specialized, sequenced prompts
5. **Examples are documentation** — Few-shot examples must be diverse and consistent
6. **Every token costs** — Optimize prompt length; 30% shorter often performs equally
7. **Cache aggressively** — 30–60% of production traffic is typically cacheable
8. **Temperature is a dial, not a switch** — Match temperature precisely to task type
9. **Model-specific optimization** — Same prompt can differ 30–40% across models
10. **ReAct traces are your debugger** — Log every thought/action/observation in production
11. **Version prompts like code** — Semantic versioning, changelogs, canary deploys
12. **Async everything parallelizable** — Sequential calls for parallelizable work = wasted latency
13. **Design for failure** — Retry logic, circuit breakers, and fallback models are mandatory
14. **Constitutional AI scales** — Principle-based critique is more consistent than open-ended
15. **The eval set is ground truth** — Protect it like production data; never train on it


In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════╗
║     QUICK REFERENCE — Prompt Engineering Cheatsheet (Part 2)         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  TECHNIQUE SELECTION GUIDE:                                           ║
║                                                                       ║
║  Need external data / tools           → ReAct Pattern                ║
║  Need to auto-optimize a prompt       → Meta-Prompting               ║
║  Output quality needs improvement     → Self-Critique Pipeline       ║
║  Need consistent safety/compliance    → Constitutional AI            ║
║  Before deploying any prompt change   → Eval Suite + A/B Test       ║
║  Picking between 2+ prompt versions   → Regression Suite             ║
║  High traffic, repeated inputs        → SmartPromptManager + Cache  ║
║  Need to process hundreds of items    → Async batch processing       ║
║  Prompt doing too many things at once → Break into prompt chain      ║
║                                                                       ║
║  REACT LOOP TEMPLATE:                                                 ║
║  Thought: [reason] → Action: tool("input") → Observe → repeat →     ║
║  Final Answer: [complete response]                                    ║
║                                                                       ║
║  PRODUCTION DEPLOY CHECKLIST:                                         ║
║  □ Eval suite >= 30 cases (easy + edge + adversarial)               ║
║  □ A/B test at least 2 prompt versions with stats                   ║
║  □ Regression guard in CI/CD (block if score drops >3%)             ║
║  □ Semantic caching enabled (expect 30-60% hit rate)                ║
║  □ Cost and P95 latency tracked per prompt version                  ║
║  □ Retry logic + fallback model configured                           ║
║  □ Async processing for any batch workloads                         ║
║  □ Prompt version changelog maintained in code review               ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")
